In [ ]:
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.types import interrupt, Command
import keys


c:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process(name : str, age : int) -> str:
    """Use this tool to process a person's admission request.
        You must provide:
        - name (str): Full name of the applicant
        - age (int): Age of the applicant
    """

    if age < 18:
        approval = interrupt(f"Do you want to approve the admission of {name} who is {age} years old? (yes/no):")
        if approval is True:
             return  f"Processed the admission of {name}, who is minor! "
        else:
             return  f"Sorry! Cannot process the admission of {name}, who is a minor! "
    else:
        return  f"Processed the admission of {name}! "

In [3]:
tools = [process]

In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    huggingfacehub_api_token=keys.HUGGINGFACE_KEY,
    task="text-generation"   
)

chat = ChatHuggingFace(llm=llm)
llm_with_tools = chat.bind_tools(tools)

# System message
sys_msg = SystemMessage(content="You are a helpful assistant. If the user asks to process admission, you MUST use the process tool.")

In [5]:
# Node
def call_llm(state: MessagesState) -> MessagesState:
    result = llm_with_tools.invoke([sys_msg] + state["messages"])
    return  {"messages": [result]}


In [17]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("call_llm", call_llm)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine the control flow
builder.add_edge(START, "call_llm")
builder.add_conditional_edges(
    "call_llm",
    tools_condition
)

builder.add_edge("tools", "call_llm")
builder.add_edge("call_llm", END)

In [18]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [20]:
messages = [HumanMessage(
        content="Process admission of Scott Bentton who is 15 years old")]

In [21]:
# Thread
thread = {"configurable": {"thread_id": "1"}}
response = graph.invoke( {"messages" : messages}, thread)

In [22]:
if "__interrupt__" in response:
    question = response['__interrupt__'][0].value   # Prompt
    user_approval = input(question)

    # Check approval
    response = graph.invoke(Command(resume=user_approval.lower() == "yes"), thread)


In [23]:
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Process admission of Scott Bentton who is 15 years old
================================== Ai Message ==================================
Tool Calls:
  process (chatcmpl-tool-c5e74ec31f9940b98e5cb5e9e1859da0)
 Call ID: chatcmpl-tool-c5e74ec31f9940b98e5cb5e9e1859da0
  Args:
    name: Scott Bentton
    age: 15
================================= Tool Message =================================
Name: process

Sorry! Cannot process the admission of Scott Bentton, who is a minor! 
================================== Ai Message ==================================

I’m sorry, but I can’t process an admission for Scott Bentton because he is under the required age limit. If you have another applicant who meets the age requirements, I’d be happy to help with their admission.
